# Mini Project 1 — Beijing Air Quality Baseline

This notebook establishes the baseline machine learning solution for the
Beijing air quality dataset.

The goal is to predict the PM2.5 concentration for the next hour using
historical air-quality and weather observations.



In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## 1. Project paths

Define the project directories and the location of the raw dataset.


In [2]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")


Project root: e:\MLops - كورس\mini projects\air-quality-mlops
Raw data directory: e:\MLops - كورس\mini projects\air-quality-mlops\data\raw


## 2. Load and combine the station datasets

The raw data contains one CSV file for each Beijing monitoring station.
We will load the files and combine them into a single DataFrame.


In [3]:
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))
print(f"Number of CSV files found: {len(csv_files)}")
for file in csv_files:
    print(file.name)


Number of CSV files found: 12
PRSA_Data_Aotizhongxin_20130301-20170228.csv
PRSA_Data_Changping_20130301-20170228.csv
PRSA_Data_Dingling_20130301-20170228.csv
PRSA_Data_Dongsi_20130301-20170228.csv
PRSA_Data_Guanyuan_20130301-20170228.csv
PRSA_Data_Gucheng_20130301-20170228.csv
PRSA_Data_Huairou_20130301-20170228.csv
PRSA_Data_Nongzhanguan_20130301-20170228.csv
PRSA_Data_Shunyi_20130301-20170228.csv
PRSA_Data_Tiantan_20130301-20170228.csv
PRSA_Data_Wanliu_20130301-20170228.csv
PRSA_Data_Wanshouxigong_20130301-20170228.csv


In [4]:
dataframes = []
for file in csv_files:
    station_df = pd.read_csv(file)
    dataframes.append(station_df)
df = pd.concat(dataframes, ignore_index=True)
print(f"Combined dataset shape: {df.shape}")



Combined dataset shape: (420768, 18)


## 3. Prepare the time column

Combine the year, month, day, and hour columns into a single datetime column.
This will allow us to create time-based features and perform a chronological
train/test split.


In [5]:
df["datetime"] = pd.to_datetime(
    df[["year", "month", "day", "hour"]]
)
df = df.sort_values(
    ["station", "datetime"]
).reset_index(drop=True)
df[["station", "datetime", "PM2.5"]].head()


,station,datetime,PM2.5
0,Aotizhongxin,2013-03-01 00:00:00,4.0
1,Aotizhongxin,2013-03-01 01:00:00,8.0
2,Aotizhongxin,2013-03-01 02:00:00,7.0
3,Aotizhongxin,2013-03-01 03:00:00,6.0
4,Aotizhongxin,2013-03-01 04:00:00,3.0


## 4. Create the prediction target

The target is the PM2.5 concentration one hour into the future.

For each station, the target is created by shifting the PM2.5 value by one
hour. The timestamp is checked to make sure the next observation is exactly
one hour later.


In [6]:
next_pm25 = (
    df.groupby("station")["PM2.5"]
      .shift(-1)
)
next_time = (
    df.groupby("station")["datetime"]
      .shift(-1)
)
df["PM2.5_next"] = next_pm25.where(
    next_time.eq(df["datetime"] + pd.Timedelta(hours=1))
)
print(f"Rows: {len(df):,}")
print(f"Missing target values: {df["PM2.5_next"].isna().sum():,}")


Rows: 420,768
Missing target values: 8,751


## 5. Create baseline features

Use the current observation together with a small set of historical PM2.5
and time-based features.

The features are intentionally kept simple because this notebook is the
baseline solution.


In [7]:
df["PM2.5_lag_1"] = (
    df.groupby("station")["PM2.5"].shift(1)
)
df["PM2.5_lag_3"] = (
    df.groupby("station")["PM2.5"].shift(3)
)
df["PM2.5_lag_24"] = (
    df.groupby("station")["PM2.5"].shift(24)
)
df["day_of_week"] = df["datetime"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)


## 6. Define the training and test periods

Because this is a time-series prediction problem, the data is split
chronologically rather than randomly.

The model is trained on earlier observations and evaluated on later
observations.


In [8]:
cutoff = pd.Timestamp("2016-05-12 19:00:00")
train_df = df[
    (df["datetime"] <= cutoff) &
    (df["PM2.5_next"].notna())
].copy()
test_df = df[
    (df["datetime"] > cutoff) &
    (df["PM2.5_next"].notna())
].copy()
print(f"Training rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")


Training rows: 329,605
Test rows: 82,412


## 7. Define features and target

Remove columns that should not be used directly by the model.

The target column is `PM2.5_next`.


In [9]:
drop_columns = [
    "No",
    "datetime",
    "PM2.5_next",
]
X_train = train_df.drop(columns=drop_columns)
y_train = train_df["PM2.5_next"]
X_test = test_df.drop(columns=drop_columns)
y_test = test_df["PM2.5_next"]
print(f"Training features: {X_train.shape}")
print(f"Test features: {X_test.shape}")


Training features: (329605, 26)
Test features: (82412, 26)


In [10]:
categorical_features = [
    "wd",
    "station",
]
numeric_features = [
    column
    for column in X_train.columns
    if column not in categorical_features
]
print("Numeric features:")
print(numeric_features)
print("\nCategorical features:")
print(categorical_features)


Numeric features:
['year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM', 'PM2.5_lag_1', 'PM2.5_lag_3', 'PM2.5_lag_24', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']

Categorical features:
['wd', 'station']


## 8. Build the preprocessing pipeline

Numeric features use median imputation followed by standardization.

Categorical features use most-frequent imputation followed by one-hot
encoding.

The preprocessing pipeline is fitted only on the training data.


In [11]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)


## 9. Train the baseline model

Use a Random Forest Regressor as the initial baseline model.

The model and preprocessing steps are combined into a single pipeline.


In [12]:
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=30,
                max_depth=15,
                min_samples_leaf=10,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

baseline_model.fit(X_train, y_train)

print("Baseline model training completed.")

Baseline model training completed.


## 10. Evaluate the baseline model

Evaluate the model on the chronological test set using Mean Absolute Error
(MAE) and Root Mean Squared Error (RMSE).


In [13]:
y_pred = baseline_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Validation MAE:  {mae:.4f}")
print(f"Validation RMSE: {rmse:.4f}")


Validation MAE:  9.5479
Validation RMSE: 18.2665


## 11. Save the baseline model

Save the complete preprocessing and model pipeline so that the exact same
transformation steps can be reused later.

This saved artifact will become the starting point for the MLOps packaging
work.


In [14]:
baseline_model_path = MODEL_DIR / "baseline.pkl"
joblib.dump(
    baseline_model,
    baseline_model_path,
    compress=3,
)
print(f"Baseline model saved to: {baseline_model_path}")


Baseline model saved to: e:\MLops - كورس\mini projects\air-quality-mlops\models\baseline.pkl


## Baseline result

The baseline model has been trained, evaluated, and saved.

The validation MAE recorded here will be used later as the reference point
when the project is refactored and converted into an MLOps-oriented system.
